In [2]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2
import os

os.makedirs('../data/clean', exist_ok=True)

In [22]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name.lower() in ["notebooks", "eda"]:
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Project root:", PROJECT_ROOT)

Project root: C:\dishu\Cleartrace


> Importing required libraries and creating a clean folder to save the clean  csv file

In [3]:
df = pd.read_csv(r"C:\dishu\Cleartrace\data\processed\delhi_aqi_90d.csv")

df['timestamp_hour'] = pd.to_datetime(df['timestamp_hour'])
df = df.sort_values(['station_name', 'timestamp_hour']).reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Dtypes of columns to fix:")
print(df[['wind_direction_10m', 'relative_humidity_2m']].dtypes)

Shape: (82004, 44)
Dtypes of columns to fix:
wind_direction_10m      int64
relative_humidity_2m    int64
dtype: object


## Task 1 — Datatype Conversion

wind_direction_10m and relative_humidity_2m are stored as int64.
Converting to float64 for two reasons:
- Consistency with other meteorological columns.
- wind_direction_10m needs float64 for sin/cos calculations.

In [4]:
df['wind_direction_10m'] = df['wind_direction_10m'].astype(float)
df['relative_humidity_2m'] = df['relative_humidity_2m'].astype(float)

print(df[['wind_direction_10m', 'relative_humidity_2m']].dtypes)

wind_direction_10m      float64
relative_humidity_2m    float64
dtype: object


## Task 2 - To add reliability column:
- If the valid AQI hours> 50 % of total hours,  we flag the station as reliable
- If not, we dont

In [5]:
reliability = df.groupby('station_name')['aqi_calculation_valid'].mean()

df['station_reliability'] = df['station_name'].map(reliability) > 0.5

print("Reliability per station:")
print(reliability.sort_values())

Reliability per station:
station_name
Chandni Chowk, Delhi - IITM                         0.077386
IHBAS, Dilshad Garden,New Delhi - CPCB              0.382298
NSIT Dwarka, Delhi - CPCB                           0.445783
New Moti Bagh, Delhi - MHUA                         0.539388
Alipur, Delhi - DPCC                                0.700185
Sonia Vihar, Delhi - DPCC                           0.715941
Anand Vihar, New Delhi - DPCC                       0.721501
Mandir Marg, New Delhi - DPCC                       0.741891
North Campus, DU, Delhi - IMD                       0.745134
Vivek Vihar, Delhi - DPCC                           0.745598
Burari Crossing, New Delhi - IMD                    0.753012
Dr. Karni Singh Shooting Range, Delhi - DPCC        0.753939
Najafgarh, Delhi - DPCC                             0.764597
Jahangirpuri, Delhi - DPCC                          0.765060
ITO, New Delhi - CPCB                               0.765524
Punjabi Bagh, Delhi - DPCC                     

## Findings:
> We conclude that the 3 stations-
- Chandni Chowk, Delhi - IITM
- IHBAS, Dilshad Garden,New Delhi - CPCB
- NSIT Dwarka, Delhi - CPCB
> are  marked as unreliable, which confirms our findings from missingno heatmap

## Task 3 — Missing Value Imputation

Strategy based on gap size:
- Gaps ≤ 6 hours → linear interpolation per station (limit=6)
- Gaps > 6 hours → proxy fill from nearest reliable station

Linear interpolation draws a straight line between last known 
and next known value — valid for short gaps since pollution 
changes gradually. Long gaps need proxy fill from nearby stations.

Chandni Chowk's 2-month gaps remain NaN intentionally — 
station is already flagged as low reliability.

In [6]:
pollutants = ['pm25', 'pm10', 'no2', 'co', 'so2', 'o3']

df_clean = df.copy()

for col in pollutants:
    df_clean[col] = (
        df_clean.groupby('station_name')[col]
        .transform(lambda x: x.interpolate(method='linear', limit=6))
    )

print("Missing values after interpolation:")
print(df_clean[pollutants].isnull().sum())
print(f"\nBefore: {df[pollutants].isnull().sum().sum()}")
print(f"After:  {df_clean[pollutants].isnull().sum().sum()}")

Missing values after interpolation:
pm25     6408
pm10     7842
no2      6447
co       6588
so2     20181
o3       7859
dtype: int64

Before: 129998
After:  55325


In [7]:
remaining = df_clean[pollutants].isnull().sum()
print("Still needs proxy fill (gaps > 6 hours):")
print(remaining)
print(f"\nTotal remaining: {remaining.sum()}")

Still needs proxy fill (gaps > 6 hours):
pm25     6408
pm10     7842
no2      6447
co       6588
so2     20181
o3       7859
dtype: int64

Total remaining: 55325


## Task 4 — Proxy Fill for Long Gaps (> 6 hours)

For gaps that interpolation couldn't fill, we use readings from 
the nearest reliable station at the same timestamp.

Steps:
1. Build nearest reliable neighbour lookup using Haversine distance
2. For each missing value, find that timestamp in the nearest reliable station
3. Fill with that station's value

Only reliable stations (station_reliability = True) are used as proxies.

In [8]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# Get one row per station with lat/lon and reliability
station_info = df_clean.groupby('station_name').agg(
    latitude=('latitude', 'first'),
    longitude=('longitude', 'first'),
    reliable=('station_reliability', 'first')
).reset_index()

reliable_stations = station_info[station_info['reliable'] == True]

# For each station, find its nearest reliable neighbour
nearest_reliable = {}

for _, row in station_info.iterrows():
    station = row['station_name']
    distances = reliable_stations[reliable_stations['station_name'] != station].copy()
    distances['dist'] = distances.apply(
        lambda r: haversine(row['latitude'], row['longitude'], 
                           r['latitude'], r['longitude']), axis=1
    )
    nearest = distances.loc[distances['dist'].idxmin(), 'station_name']
    nearest_reliable[station] = nearest

print("Nearest reliable station per station:")
for k, v in nearest_reliable.items():
    print(f"  {k[:35]:35s} → {v}")

Nearest reliable station per station:
  Alipur, Delhi - DPCC                → Narela, Delhi - DPCC
  Anand Vihar, New Delhi - DPCC       → Vivek Vihar, Delhi - DPCC
  Ashok Vihar, Delhi - DPCC           → Wazirpur, Delhi - DPCC
  Aya Nagar, New Delhi - IMD          → Sri Aurobindo Marg, Delhi - DPCC
  Bawana, Delhi - DPCC                → DTU, New Delhi - CPCB
  Burari Crossing, New Delhi - IMD    → Jahangirpuri, Delhi - DPCC
  CRRI Mathura Road, New Delhi - IMD  → Okhla Phase-2, Delhi - DPCC
  Chandni Chowk, Delhi - IITM         → ITO, New Delhi - CPCB
  DTU, New Delhi - CPCB               → Rohini, Delhi - DPCC
  Dr. Karni Singh Shooting Range, Del → Okhla Phase-2, Delhi - DPCC
  Dwarka-Sector 8, Delhi - DPCC       → IGI Airport (T3), Delhi - IMD
  IGI Airport (T3), Delhi - IMD       → Dwarka-Sector 8, Delhi - DPCC 
  IHBAS, Dilshad Garden,New Delhi - C → Vivek Vihar, Delhi - DPCC
  ITO, New Delhi - CPCB               → Major Dhyan Chand National Stadium, Delhi - DPCC
  Jahangirpuri,

In [11]:
from math import radians, sin, cos, sqrt, atan2

MIN_VALID_RATIO = 0.50
MIN_NON_NULL_VALUES = 500

def haversine_km(lat1, lon1, lat2, lon2):
    radius = 6371

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return radius * c


df_clean["timestamp_hour"] = pd.to_datetime(df_clean["timestamp_hour"], errors="coerce")

station_meta = (
    df_clean
    .groupby("station_name", as_index=False)
    .agg(
        latitude=("latitude", "first"),
        longitude=("longitude", "first")
    )
)

station_meta = station_meta.dropna(subset=["latitude", "longitude"]).copy()

proxy_fill_logs = []

for col in pollutants:
    station_quality = (
        df_clean
        .groupby("station_name")[col]
        .agg(
            non_null_count=lambda s: s.notna().sum(),
            valid_ratio=lambda s: s.notna().mean()
        )
        .reset_index()
    )

    reliable_stations = station_quality[
        (station_quality["valid_ratio"] >= MIN_VALID_RATIO)
        & (station_quality["non_null_count"] >= MIN_NON_NULL_VALUES)
    ]["station_name"].tolist()

    pivot = df_clean.pivot_table(
        index="timestamp_hour",
        columns="station_name",
        values=col,
        aggfunc="first"
    )

    for station in df_clean["station_name"].dropna().unique():
        station_rows = df_clean.index[
            (df_clean["station_name"] == station)
            & (df_clean[col].isna())
        ]

        if len(station_rows) == 0:
            continue

        candidate_stations = [
            candidate for candidate in reliable_stations
            if candidate != station and candidate in pivot.columns
        ]

        if len(candidate_stations) == 0:
            proxy_fill_logs.append(
                {
                    "pollutant": col,
                    "station": station,
                    "proxy_station": None,
                    "distance_km": np.nan,
                    "missing_before": len(station_rows),
                    "proxy_values_available": 0,
                    "filled_count": 0,
                    "reason": "no reliable candidate"
                }
            )
            continue

        target_meta = station_meta[station_meta["station_name"] == station]

        if target_meta.empty:
            proxy_fill_logs.append(
                {
                    "pollutant": col,
                    "station": station,
                    "proxy_station": None,
                    "distance_km": np.nan,
                    "missing_before": len(station_rows),
                    "proxy_values_available": 0,
                    "filled_count": 0,
                    "reason": "missing station coordinates"
                }
            )
            continue

        target_lat = target_meta["latitude"].iloc[0]
        target_lon = target_meta["longitude"].iloc[0]

        candidate_meta = station_meta[
            station_meta["station_name"].isin(candidate_stations)
        ].copy()

        candidate_meta["distance_km"] = candidate_meta.apply(
            lambda row: haversine_km(
                target_lat,
                target_lon,
                row["latitude"],
                row["longitude"]
            ),
            axis=1
        )

        candidate_meta = candidate_meta.sort_values("distance_km")

        proxy_station = candidate_meta["station_name"].iloc[0]
        proxy_distance = candidate_meta["distance_km"].iloc[0]

        missing_timestamps = df_clean.loc[station_rows, "timestamp_hour"]

        proxy_values = pivot[proxy_station].reindex(missing_timestamps)

        proxy_values_available = proxy_values.notna().sum()

        df_clean.loc[station_rows, col] = proxy_values.to_numpy()

        filled_count = df_clean.loc[station_rows, col].notna().sum()

        proxy_fill_logs.append(
            {
                "pollutant": col,
                "station": station,
                "proxy_station": proxy_station,
                "distance_km": round(proxy_distance, 2),
                "missing_before": len(station_rows),
                "proxy_values_available": int(proxy_values_available),
                "filled_count": int(filled_count),
                "reason": "filled from pollutant-specific nearest reliable station"
            }
        )

proxy_fill_report = pd.DataFrame(proxy_fill_logs)

print("Missing values after pollutant-specific proxy fill:")
print(df_clean[pollutants].isnull().sum())

print("\nTotal remaining:")
print(df_clean[pollutants].isnull().sum().sum())

print("\nProxy fill summary by pollutant:")
display(
    proxy_fill_report
    .groupby("pollutant")
    .agg(
        total_missing_attempted=("missing_before", "sum"),
        total_proxy_values_available=("proxy_values_available", "sum"),
        total_filled=("filled_count", "sum"),
        stations_attempted=("station", "nunique")
    )
    .reset_index()
)

print("\nProxy fill report:")
display(proxy_fill_report.sort_values(["pollutant", "filled_count"], ascending=[True, False]))

Missing values after pollutant-specific proxy fill:
pm25    2753
pm10    2826
no2     2761
co      2825
so2     3121
o3      2864
dtype: int64

Total remaining:
17150

Proxy fill summary by pollutant:


,pollutant,total_missing_attempted,total_proxy_values_available,total_filled,stations_attempted
0,co,2920,95,95,38
1,no2,2843,82,82,38
2,o3,7859,4995,4995,38
3,pm10,2854,28,28,38
4,pm25,2760,7,7,38
5,so2,12073,8952,8952,38



Proxy fill report:


,pollutant,station,proxy_station,distance_km,missing_before,proxy_values_available,filled_count,reason
121,co,"Chandni Chowk, Delhi - IITM","ITO, New Delhi - CPCB",3.41,114,42,42,filled from pollutant-specific nearest reliabl...
148,co,"Sonia Vihar, Delhi - DPCC","Burari Crossing, New Delhi - IMD",5.00,80,11,11,filled from pollutant-specific nearest reliabl...
137,co,"Nehru Nagar, Delhi - DPCC","Jawaharlal Nehru Stadium, Delhi - DPCC",2.13,79,9,9,filled from pollutant-specific nearest reliabl...
150,co,"Vivek Vihar, Delhi - DPCC","IHBAS, Dilshad Garden,New Delhi - CPCB",1.58,83,7,7,filled from pollutant-specific nearest reliabl...
123,co,"Dr. Karni Singh Shooting Range, Delhi - DPCC","Okhla Phase-2, Delhi - DPCC",3.64,76,6,6,filled from pollutant-specific nearest reliabl...
...,...,...,...,...,...,...,...,...
160,so2,"DTU, New Delhi - CPCB","Rohini, Delhi - DPCC",2.12,74,0,0,filled from pollutant-specific nearest reliabl...
161,so2,"Dr. Karni Singh Shooting Range, Delhi - DPCC","Okhla Phase-2, Delhi - DPCC",3.64,76,0,0,filled from pollutant-specific nearest reliabl...
166,so2,"Jahangirpuri, Delhi - DPCC","Burari Crossing, New Delhi - IMD",3.08,95,0,0,filled from pollutant-specific nearest reliabl...
171,so2,"Mundka, Delhi - DPCC","Punjabi Bagh, Delhi - DPCC",5.44,76,0,0,filled from pollutant-specific nearest reliabl...


## Findings from proxy filling approach:
- PM25 and PM10 were barely filled with proxy reliable station values, this might suggest that there is a citywide outrage/unavailability for these pollutants at some specific hours.
- SO2 and 03 were filled with tremendous amounts, thus not suggesting any anamoly in station wide providement of their values.

In [13]:
df_before_proxy = df_clean.copy()

In [14]:
pollutants = ["pm25", "pm10", "no2", "co", "so2", "o3"]

citywide_missing_summary = []

for col in pollutants:
    temp = (
        df_before_proxy
        .groupby("timestamp_hour")
        .agg(
            total_stations=("station_name", "nunique"),
            missing_stations=(col, lambda s: s.isna().sum())
        )
        .reset_index()
    )

    temp["missing_station_ratio"] = temp["missing_stations"] / temp["total_stations"]

    citywide_missing_summary.append(
        {
            "pollutant": col,
            "hours_50pct_missing": (temp["missing_station_ratio"] >= 0.50).sum(),
            "hours_75pct_missing": (temp["missing_station_ratio"] >= 0.75).sum(),
            "hours_90pct_missing": (temp["missing_station_ratio"] >= 0.90).sum(),
            "max_missing_ratio": round(temp["missing_station_ratio"].max(), 3),
            "mean_missing_ratio": round(temp["missing_station_ratio"].mean(), 3)
        }
    )

citywide_missing_summary = pd.DataFrame(citywide_missing_summary)

display(citywide_missing_summary)

,pollutant,hours_50pct_missing,hours_75pct_missing,hours_90pct_missing,max_missing_ratio,mean_missing_ratio
0,pm25,72,70,70,1.0,0.034
1,pm10,72,70,70,1.0,0.034
2,no2,72,70,70,1.0,0.034
3,co,72,70,70,1.0,0.034
4,so2,74,72,72,1.0,0.038
5,o3,72,70,70,1.0,0.035


- Most PM2.5, PM10, NO2, and CO missing values occur during synchronized city-wide missing periods, so nearest-station proxy imputation is not effective for these pollutants.

- SO2 and O3 contain additional station-specific missingness, so nearest-station proxy imputation can fill many of their gaps. However, these proxy-filled values should be flagged because they are estimated, not directly observed.

In [15]:
pollutants = ["pm25", "pm10", "no2", "co", "so2", "o3"]

outage_rows = []

for col in pollutants:
    temp = (
        df_before_proxy
        .groupby("timestamp_hour")
        .agg(
            total_stations=("station_name", "nunique"),
            missing_stations=(col, lambda s: s.isna().sum())
        )
        .reset_index()
    )

    temp["missing_station_ratio"] = temp["missing_stations"] / temp["total_stations"]
    temp["pollutant"] = col

    outage_rows.append(temp[temp["missing_station_ratio"] >= 0.90])

outage_df = pd.concat(outage_rows, ignore_index=True)

display(
    outage_df
    .sort_values(["timestamp_hour", "pollutant"])
    .head(100)
)

,timestamp_hour,total_stations,missing_stations,missing_station_ratio,pollutant
210,2026-04-16 20:00:00,38,38,1.0,co
140,2026-04-16 20:00:00,38,38,1.0,no2
352,2026-04-16 20:00:00,38,38,1.0,o3
70,2026-04-16 20:00:00,38,38,1.0,pm10
0,2026-04-16 20:00:00,38,38,1.0,pm25
...,...,...,...,...,...
85,2026-05-08 00:00:00,38,38,1.0,pm10
15,2026-05-08 00:00:00,38,38,1.0,pm25
297,2026-05-08 00:00:00,38,38,1.0,so2
226,2026-06-23 16:00:00,38,38,1.0,co


In [16]:
outage_summary_by_date = (
    outage_df
    .assign(date=outage_df["timestamp_hour"].dt.date)
    .groupby(["date", "pollutant"])
    .size()
    .reset_index(name="high_missing_hours")
)

display(outage_summary_by_date)

,date,pollutant,high_missing_hours
0,2026-04-16,co,2
1,2026-04-16,no2,2
2,2026-04-16,o3,2
3,2026-04-16,pm10,2
4,2026-04-16,pm25,2
5,2026-04-16,so2,2
6,2026-05-02,co,1
7,2026-05-02,no2,1
8,2026-05-02,o3,1
9,2026-05-02,pm10,1


## Findings:
- There is day wise outage for the pollutants for specific hours ranging from 1 to 24
- Majority of the pollutants are missing together for all the stations(majority).
- Dropping such rows is a better option than proxy filling
  

In [17]:
pollutants = ["pm25", "pm10", "no2", "co", "so2", "o3"]

outage_flags = []

for col in pollutants:
    temp = (
        df_before_proxy
        .groupby("timestamp_hour")
        .agg(
            total_stations=("station_name", "nunique"),
            missing_stations=(col, lambda s: s.isna().sum())
        )
        .reset_index()
    )

    temp["missing_station_ratio"] = temp["missing_stations"] / temp["total_stations"]
    temp["pollutant"] = col
    temp["is_citywide_missing"] = temp["missing_station_ratio"] >= 0.90

    outage_flags.append(temp)

outage_flags = pd.concat(outage_flags, ignore_index=True)

citywide_outage_hours = (
    outage_flags
    .groupby("timestamp_hour")
    .agg(
        pollutants_citywide_missing=("is_citywide_missing", "sum")
    )
    .reset_index()
)

citywide_outage_hours["is_citywide_outage_hour"] = (
    citywide_outage_hours["pollutants_citywide_missing"] >= 4
)

outage_timestamps = citywide_outage_hours.loc[
    citywide_outage_hours["is_citywide_outage_hour"],
    "timestamp_hour"
]

print("City-wide outage hours detected:", len(outage_timestamps))

display(
    citywide_outage_hours[
        citywide_outage_hours["is_citywide_outage_hour"]
    ].head(50)
)

City-wide outage hours detected: 70


,timestamp_hour,pollutants_citywide_missing,is_citywide_outage_hour
143,2026-04-16 20:00:00,6,True
144,2026-04-16 21:00:00,6,True
529,2026-05-02 22:00:00,6,True
602,2026-05-05 23:00:00,6,True
603,2026-05-06 00:00:00,6,True
604,2026-05-06 01:00:00,6,True
605,2026-05-06 02:00:00,6,True
606,2026-05-06 03:00:00,6,True
607,2026-05-06 04:00:00,6,True
608,2026-05-06 05:00:00,6,True


In [18]:
rows_before = len(df_clean)

df_clean = df_clean[
    ~df_clean["timestamp_hour"].isin(outage_timestamps)
].copy()

rows_after = len(df_clean)

print("Rows before dropping outage hours:", rows_before)
print("Rows after dropping outage hours:", rows_after)
print("Rows dropped:", rows_before - rows_after)

Rows before dropping outage hours: 82004
Rows after dropping outage hours: 79344
Rows dropped: 2660


## Findings:
City-wide outage analysis showed that most high-missingness periods were clustered on specific dates, especially 2026-06-23 to 2026-06-25. Since these missing periods affected nearly all stations and pollutants simultaneously, they were treated as provider/data-availability outages rather than station-level gaps. These timestamps were removed from the cleaned master dataset instead of being imputed.


In [19]:
print("Missing values after dropping city-wide outage hours:")
print(df_clean[pollutants].isnull().sum())

print("\nTotal remaining missing pollutant values:")
print(df_clean[pollutants].isnull().sum().sum())

print("\nMissing percentage:")
print((df_clean[pollutants].isnull().mean() * 100).round(2))

Missing values after dropping city-wide outage hours:
pm25     93
pm10    169
no2     104
co      168
so2     464
o3      206
dtype: int64

Total remaining missing pollutant values:
1204

Missing percentage:
pm25    0.12
pm10    0.21
no2     0.13
co      0.21
so2     0.58
o3      0.26
dtype: float64


In [20]:
master_clean = df_before_proxy[
    ~df_before_proxy["timestamp_hour"].isin(outage_timestamps)
].copy()

print("Master clean shape:", master_clean.shape)

print("\nRemaining missing values:")
print(master_clean[pollutants].isnull().sum())

print("\nRemaining missing percentage:")
print((master_clean[pollutants].isnull().mean() * 100).round(2))


Master clean shape: (79344, 45)

Remaining missing values:
pm25     93
pm10    169
no2     104
co      168
so2     464
o3      206
dtype: int64

Remaining missing percentage:
pm25    0.12
pm10    0.21
no2     0.13
co      0.21
so2     0.58
o3      0.26
dtype: float64


In [23]:
output_path = PROJECT_ROOT / "data" / "processed" / "master_clean.csv"
master_clean.to_csv(output_path, index=False)

print("Saved:", output_path)

Saved: C:\dishu\Cleartrace\data\processed\master_clean.csv


In [24]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name.lower() in ["notebooks", "eda"]:
    PROJECT_ROOT = PROJECT_ROOT.parent

master_path = PROJECT_ROOT / "data" / "processed" / "master_clean.csv"

master_clean = pd.read_csv(master_path)
master_clean["timestamp_hour"] = pd.to_datetime(master_clean["timestamp_hour"], errors="coerce")

print("Shape:", master_clean.shape)
print("Stations:", master_clean["station_name"].nunique())
print("Timestamp min:", master_clean["timestamp_hour"].min())
print("Timestamp max:", master_clean["timestamp_hour"].max())

print("\nMissing pollutant values:")
pollutants = ["pm25", "pm10", "no2", "co", "so2", "o3"]
print(master_clean[pollutants].isnull().sum())

print("\nMissing pollutant percentage:")
print((master_clean[pollutants].isnull().mean() * 100).round(3))

print("\nDuplicate station-hour rows:")
print(master_clean.duplicated(subset=["station_name", "timestamp_hour"]).sum())

print("\nAQI validity:")
print(master_clean["aqi_calculation_valid"].value_counts(dropna=False))

print("\nDominant pollutant:")
print(master_clean["dominant_pollutant"].value_counts(dropna=False))

Shape: (79344, 45)
Stations: 38
Timestamp min: 2026-04-10 21:00:00
Timestamp max: 2026-07-09 18:00:00

Missing pollutant values:
pm25     93
pm10    169
no2     104
co      168
so2     464
o3      206
dtype: int64

Missing pollutant percentage:
pm25    0.117
pm10    0.213
no2     0.131
co      0.212
so2     0.585
o3      0.260
dtype: float64

Duplicate station-hour rows:
0

AQI validity:
aqi_calculation_valid
True     61199
False    18145
Name: count, dtype: int64

Dominant pollutant:
dominant_pollutant
pm10    48467
NaN     18145
pm25     6135
no2      4751
so2       683
co        643
o3        520
Name: count, dtype: int64
